In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!nvidia-smi

Thu Mar 12 16:56:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:61:00.0 Off |                    0 |
| N/A   27C    P8             32W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Put project root on sys.path so "source" is importable
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> root/
print(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

/home/mpominova/lcms-foundation-model


In [4]:
import os
import yaml
import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import pytorch_lightning as L
from tqdm import tqdm
from depthcharge.data import SpectrumDataset, spectra_to_df, preprocessing
# from depthcharge.tokenizers import Tokenizer
from torch.utils.data import DataLoader

from source.dataset import LanceMapDataset, RunDataset
from source.model import MS1Encoder
from source.scheduler import CosineWarmupScheduler
from source.config import ExperimentConfig, DataConfig, ModelConfig, OptimizerConfig, TrainingConfig

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import numpy as np
import pandas as pd # DEBUG
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as L
from depthcharge.encoders import PeakEncoder, PositionalEncoder
from depthcharge.transformers import SpectrumTransformerEncoder

from IPython.display import clear_output # DEBUG
pd.set_option('display.max_rows', 500) # DEBUG

class MS1Encoder(L.LightningModule):
    def __init__(
        self,
        d_model=128,
        nhead=8,
        dim_feedforward=512,
        n_layers=4,
        dropout=0.1,
        n_bins=2000,
        bin_mz_min=0,
        bin_mz_max=2000,
        masked_peaks_fraction=0.3,
        mask_proportional=True,
        lr=5e-4,
        warmup_iters=1000,
        cosine_schedule_period_iters=32000,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.d_model = d_model
        self.nhead = nhead
        self.dim_feedforward = dim_feedforward
        self.n_layers = n_layers
        self.dropout = dropout

        self.n_bins = n_bins
        self.bin_mz_min = bin_mz_min
        self.bin_mz_max = bin_mz_max
        self.masked_peaks_fraction = masked_peaks_fraction
        self.mask_proportional = mask_proportional

        self.peak_encoder = nn.Sequential(
            PeakEncoder(
                d_model=self.d_model,
                min_mz_wavelength=0.001,
                max_mz_wavelength=10000,
                min_intensity_wavelength=1e-06,
                max_intensity_wavelength=1,
                learnable_wavelengths=False,
            ),
        )
        self.encoder = SpectrumTransformerEncoder(
            d_model=self.d_model,
            nhead=self.nhead,
            dim_feedforward=self.dim_feedforward,  # 1024,
            n_layers=self.n_layers,
            dropout=self.dropout,
            peak_encoder=self.peak_encoder,
        )

        self.head_mz = nn.Sequential(
            nn.Linear(d_model, n_bins),
        )  # outputs n_bins logits for each peak
        self.head_I = nn.Sequential(
            nn.Linear(d_model, 1),
        )  # outputs float I value for each peak

        # losses
        self.loss_mz_bin = nn.CrossEntropyLoss(reduction="mean", ignore_index=-1)
        self.loss_I = nn.MSELoss(reduction="mean")
        # metrics
        self.train_accuracy_mz_bin = torchmetrics.classification.Accuracy(task="multiclass", num_classes=self.n_bins, ignore_index=-1)
        self.val_accuracy_mz_bin = torchmetrics.classification.Accuracy(task="multiclass", num_classes=self.n_bins, ignore_index=-1)

    def get_peaks_mask(self, intensities, proportional=False):
        if proportional:
            k = int(intensities.size(1) * self.masked_peaks_fraction)
            mask = torch.zeros_like(intensities, dtype=torch.bool)
            # FIXME: assume we never have zero rows (= spectra with no peaks), otherwise will fail

            # compute sampling weights w
            I_mean = intensities.sum(dim=1) / (intensities != 0).sum(dim=1) # mean I of non-zero peaks
            w = intensities + (intensities == 0).float() * I_mean[:, None]  # weight 0s by mean I
            # sample k indices without replacement, weighted by w
            idx = torch.multinomial(w, num_samples=k, replacement=False)
            mask = mask.scatter(dim=1, index=idx.to(dtype=torch.int64), value=True) # value to write into mask (True)

        else:
            mask = torch.rand_like(intensities) < self.masked_peaks_fraction
        return mask

    def get_mz_bins(self, mz):
        # every peak with mz > bin_mz_max will belong to max bin
        mz = mz.clamp(0, self.bin_mz_max - 1)
        mz_binned = (
            ((mz - self.bin_mz_min) / (self.bin_mz_max - self.bin_mz_min) * self.n_bins)
            .floor()
            .long()
        )
        mz_binned[mz < self.bin_mz_min] = -1
        return mz_binned

    def forward(
        self,
        mzs: torch.Tensor,
        intensities: torch.Tensor,
    ):
        peak_embs, _ = self.encoder(mz_array=mzs, intensity_array=intensities)
        # drop global token
        peak_embs = peak_embs[:, 1:, :]
        return peak_embs

    # def ssl_step(self):
    #     # TODO: move here the repeated part of training & validation parts
    #     return

    def training_step(self, batch, batch_idx):
        mz = batch["mz_array"]
        I = batch["intensity_array"]

        # sample peak masks
        masks = self.get_peaks_mask(I, proportional=self.mask_proportional)

        # prepare targets (bins & I of masked peaks)
        target_mz, target_I = mz[masks], I[masks]
        # transform mz into bins (target classes C \in [0, n_bins - 1])
        target_mz_bins = self.get_mz_bins(target_mz)

        # mask input peaks with 0 (before encoding)
        masked_mz = mz * (1 - masks.float())
        # masked_I = I * (1 - masks.float()) # FIX: not mask intensities, only mz
        
        # get embeddings for all peaks
        # peak_embs = self.forward(masked_mz, masked_I)
        peak_embs = self.forward(masked_mz, I) # FIX: not mask intensities, only mz
        # select only embeddings of masked peaks
        masked_peak_embs = peak_embs[masks]
        # predict masked peaks binned mz & I
        pred_mz_bins = self.head_mz(masked_peak_embs)
        pred_I = self.head_I(masked_peak_embs).squeeze(dim=-1)

        loss_mz_bin = self.loss_mz_bin(pred_mz_bins, target_mz_bins)
        loss_I = self.loss_I(pred_I, target_I)
        loss = loss_mz_bin# + loss_I
        self.log("train_loss_mz_bin", loss_mz_bin.item())
        # self.log("train_loss_I", loss_I.item())
        self.log("train_loss", loss.item())
        # Accuracy metric for mz bin prediction
        acc_mz_bin = self.train_accuracy_mz_bin(pred_mz_bins, target_mz_bins)
        self.log("train_acc_mz_bin", acc_mz_bin.item(), prog_bar=True, on_step=True, on_epoch=False)
        return loss

    def validation_step(self, batch, batch_idx):
        # Note: validation is now always partially random (mask sampling)
        # so not identical between epochs. May want to change it (how?).
        mz = batch["mz_array"]
        I = batch["intensity_array"]

        # sample peak masks
        masks = self.get_peaks_mask(I, proportional=self.mask_proportional)

        # prepare targets (bins & I of masked peaks)
        target_mz, target_I = mz[masks], I[masks]
        # transform mz into bins (target classes C \in [0, n_bins - 1])
        target_mz_bins = self.get_mz_bins(target_mz)

        # mask input peaks with 0 (before encoding)
        masked_mz = mz * (1 - masks.float())
        # masked_I = I * (1 - masks.float()) # FIX: not mask intensities, only mz
        
        # get embeddings for all peaks
        # peak_embs = self.forward(masked_mz, masked_I)
        peak_embs = self.forward(masked_mz, I) # FIX: not mask intensities, only mz
        # select only embeddings of masked peaks
        masked_peak_embs = peak_embs[masks]
        # predict masked peaks binned mz & I
        pred_mz_bins = self.head_mz(masked_peak_embs)
        pred_I = self.head_I(masked_peak_embs).squeeze(dim=-1)

        loss_mz_bin = self.loss_mz_bin(pred_mz_bins, target_mz_bins)
        # loss_I = self.loss_I(pred_I, target_I)
        loss = loss_mz_bin# + loss_I
        self.log("val_loss_mz_bin", loss_mz_bin.item())
        # self.log("val_loss_I", loss_I.item())
        self.log("val_loss", loss.item())
        # Accuracy metric for mz bin prediction
        acc_mz_bin = self.val_accuracy_mz_bin(pred_mz_bins, target_mz_bins)
        self.log("val_acc_mz_bin", acc_mz_bin.item(), prog_bar=True, on_step=False, on_epoch=True)

        # # Display prediction sample
        # n = 30
        # mz_bins_true, I_true = target_mz_bins[:n].cpu().numpy(), target_I[:n].cpu().numpy()
        # mz_bins_pred, I_pred = pred_mz_bins[:n].argmax(dim=1).cpu().numpy(), pred_I[:n].cpu().numpy()
        # sample_df = np.column_stack((
        #     mz_bins_true.ravel(), 
        #     I_true.ravel(), 
        #     mz_bins_pred.ravel(), 
        #     # I_pred.ravel()
        # ))
        # sample_df = pd.DataFrame(
        #     sample_df, columns=[
        #         "mz_bins_true", 
        #         "I_true", 
        #         "mz_bins_pred", 
        #         # "I_pred"
        #     ]
        # )
        # display(sample_df)
        
        return loss

    def on_validation_epoch_start(self,):
        clear_output(True)
        return

    def configure_optimizers(
        self,
    ):
        """TODO."""
        optimizer = torch.optim.Adam(
            self.parameters(), lr=self.hparams.lr, betas=(0.9, 0.98)
        )
        self.lr_scheduler = CosineWarmupScheduler(
            optimizer,
            self.hparams.warmup_iters,
            self.hparams.cosine_schedule_period_iters,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": self.lr_scheduler,
                "interval": "step",
                "frequency": 1,
                "name": "cosine_warmup",
            },
        }

    def optimizer_step(self, *args, **kwargs):
        super().optimizer_step(*args, **kwargs)
        self.log("lr", self.lr_scheduler.get_last_lr()[0])

In [9]:
def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, "r") as f:
        config_dict = yaml.safe_load(f)
        config = ExperimentConfig(
            name=config_dict['name'],
            data=DataConfig(**config_dict['data']),
            model=ModelConfig(**config_dict['model']),
            optimizer=OptimizerConfig(**config_dict['optimizer']),
            training=TrainingConfig(**config_dict['training'])
        )
    return config

In [10]:
# Load training data
data_root_dir = "/mnt/data/shared/lc_ms_foundation/"
dset_name = "abele_data"
data_dir = os.path.join(data_root_dir, dset_name, "mzml")

mzml_files = os.listdir(data_dir)
print("Total N files:", len(mzml_files))

meta_df = pl.read_csv("../data/filtered_abele_metadata.csv")
meta_df = meta_df.rename({
    "characteristics[organism]": "organism",
    "comment[data file]": "data_file"
})
meta_df

Total N files: 60


organism,genus,data_file
str,str,str
"""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026"""
"""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027"""
"""Micrococcus cohnii""","""Micrococcus""","""BBM_441_P110_31_MIA_032"""
"""Micrococcus lylae""","""Micrococcus""","""BBM_446_P110_31_MIA_052"""
"""Listeria seeligeri""","""Listeria""","""BBM_446_P110_31_MIA_057"""
…,…,…
"""Buttiauxella noackiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_010_10"""
"""Buttiauxella warmboldiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_011_10"""
"""Buttiauxella ferragutiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_013_10"""


In [11]:
# preload all the data

In [12]:
# use our "default" parameters from config
config = load_config("../config.yaml")
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=2000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [13]:
preprocessing_fn = [
    preprocessing.filter_intensity(max_num_peaks=config.data.max_num_peaks),
    preprocessing.scale_intensity(scaling="root", max_intensity=1.),
]

In [14]:
# TODO: spectra_to_df не извлекает RT по дефолту, 
# - но может быть можно это переопределить через custom fields

dfs = {
    mzml_file: spectra_to_df(
        os.path.join(data_dir, mzml_file),
        metadata_df=None,
        ms_level=1,
        preprocessing_fn=preprocessing_fn,
        valid_charge=None,
        custom_fields=None,
        progress=True,
    )
    for mzml_file in mzml_files
}
len(dfs)

BBM_459_P110_31_MIA_083.mzML: 100%|██████████| 104957/104957 [01:02<00:00, 1670.18 spectra/s]


60

In [15]:
# split into train/probe_train/probe_val

# - 1) разбиение на train / probe_train / probe_val
# у нас есть 4 варианта, 

# (1) uniform split into train/val, train is both SSL train and downstream eval (probe) train,
# val is both SSL val and downstream eval (probe) val

# (2) uniform split into train/probe_train/probe_val

# (3) use a separate dataset 2 as train (for SSL), 
# and main dataset 1 uniform-split into probe_train/probe_val

# (4) сетап (3) реализуется целиком из датасета 1 
# (без добавления другого датасета 2 с потенциально совсем другим распределением)
# -- мы берем просто данные нескольких классов, которые не будут участвовать в evaluation классификации, 
# и используем их для претрейнинга, а данные небольшого количества классов (2-3) используем для валидации и evaluation
# плюс в том, что модель скорее всего будет более легко переноситься (из-за схожести данных)
# и количество классов в evaluation задаче можно уменьшить, что делает ее проще

# пока мы использовали 
# (1) (with MS1-level evaluation) и 
# (2) (with run-level evaluation)

# но вообще мы можем их варьировать, потому что (2) самый честный (если брать только один датасет), но в нем мало данных,
# (1) самый простой, и мы получаем больше данных при наличии только одного датасета, 
# а (3) также оценивает, насколько модель transferable, и в нем больше данных, 
# и можно еще параллельно увеличивать количество данных в SSL pretrain'е (learning curves!),
# но при этом есть модель недостаточно transferable, то вообще ничего оценить не получится
# (4) вероятно будет работать

In [ ]:
# get main groups (for further splitting)

In [16]:
genus_class = {
    "Listeria": 0,
    "Micrococcus": 1,
    "Shigella": 2,
    "Buttiauxella": 3,
}
meta_df = meta_df.with_columns((pl.col("data_file") + ".mzML").alias("peak_file"))
meta_df = meta_df.with_columns(
    (pl.col("genus").replace(genus_class, return_dtype=int).alias("genus_class"))
)

display(meta_df["genus_class"].value_counts())

/tmp/ipykernel_79388/2307692069.py:9: DeprecationWarning: the `return_dtype` parameter for `replace` is deprecated. Use `replace_strict` instead to set a return data type while replacing values.
(Deprecated in version 1.0.0)
  (pl.col("genus").replace(genus_class, return_dtype=int).alias("genus_class"))


genus_class,count
i64,u32
2,9
0,15
1,15
3,21


In [17]:
def get_gradient_lenght(run_name):
    run_name = run_name.split("_")
    if len(run_name) < 7:
        return ""
    return run_name[6]

In [18]:
meta_df = meta_df.with_columns(
    (pl.col("data_file").str.split("_").list[1]).alias("id1")
)
meta_df = meta_df.with_columns(
    (pl.col("data_file").str.split("_").list[5]).alias("id2")
)
display(meta_df["id2"].value_counts())

meta_df = meta_df.with_columns(
    (pl.col("data_file").map_elements(get_gradient_lenght)).alias("grad_len")
)
display(meta_df["grad_len"].value_counts())

id2,count
str,u32
"""027""",1
"""017""",3
"""052""",3
"""009""",3
"""037""",3
…,…
"""022""",3
"""011""",3
"""057""",3


/tmp/ipykernel_79388/1693487057.py:9: MapWithoutReturnDtypeWarning: 'return_dtype' of function python_udf must be set

A later expression might fail because the output type is not known. Set return_dtype=pl.self_dtype() if the type is unchanged, or set the proper output data type.
  meta_df = meta_df.with_columns(


grad_len,count
str,u32
"""30""",20
"""10""",20
"""""",20


In [19]:
# (Split 2) 
# - uniform split across all 4 classes

# we want to split runs, grouped by "id2", balanced by "genus_class"
# ids are "supergroups" -- objects with the same id should always be in the same group
# - so we only need a stratified split at the id level by target (genus class)

g = meta_df.select(["id2", "genus_class"]).group_by("id2")
sorted_ids = g.first().sort(by=["genus_class", "id2"])["id2"].to_list()
print(sorted_ids)

train_ids = sorted_ids[::3]
val_ids = [i for i in sorted_ids if i not in train_ids]

probe_train_ids = val_ids[::2]
probe_val_ids = [i for i in val_ids if i not in probe_train_ids]

len(train_ids), len(probe_train_ids), len(probe_val_ids)

['027', '037', '057', '059', '060', '072', '032', '052', '062', '083', '004', '022', '026', '002', '008', '009', '010', '011', '013', '017']


(7, 7, 6)

In [20]:
# (Split 4)

# take classes 2+3 (9+21 samples) as SSL train
# uniform split classes 0+1 (15+15 samples) into probe_train/probe_val

# train_id

In [21]:
meta_df = meta_df.with_columns(
    (pl.col("id1") + "_" + pl.col("id2")).alias("id")
)
meta_df["id"].value_counts()

id,count
str,u32
"""440_032""",2
"""437_010""",1
"""436_026""",2
"""445_059""",2
"""458_026""",2
…,…
"""437_022""",1
"""441_032""",1
"""445_052""",2


In [22]:
g = meta_df.select(["id", "genus_class"]).group_by("id")
sorted_ids = g.first().sort(by=["genus_class", "id"])["genus_class", "id"]
sorted_ids

genus_class,id
i64,str
0,"""441_027"""
0,"""445_057"""
0,"""445_059"""
0,"""445_060"""
0,"""445_072"""
…,…
3,"""437_009"""
3,"""437_010"""
3,"""437_011"""


In [23]:
train_ids = sorted_ids.filter(sorted_ids["genus_class"].is_in([2, 3]))["id"].to_list()
len(train_ids), train_ids

(20,
 ['436_004',
  '436_022',
  '436_026',
  '437_004',
  '437_022',
  '437_026',
  '436_002',
  '436_008',
  '436_009',
  '436_010',
  '436_011',
  '436_013',
  '436_017',
  '437_002',
  '437_008',
  '437_009',
  '437_010',
  '437_011',
  '437_013',
  '437_017'])

In [24]:
val_ids = [i for i in sorted_ids["id"] if i not in train_ids]

probe_train_ids = val_ids[::2]
probe_val_ids = [i for i in val_ids if i not in probe_train_ids]

len(train_ids), len(probe_train_ids), len(probe_val_ids)

(20, 10, 10)

In [25]:
display(sorted_ids.filter(sorted_ids["id"].is_in(probe_train_ids)))
display(sorted_ids.filter(sorted_ids["id"].is_in(probe_val_ids)))

genus_class,id
i64,str
0,"""441_027"""
0,"""445_059"""
0,"""445_072"""
0,"""446_059"""
0,"""458_037"""
1,"""440_032"""
1,"""445_052"""
1,"""458_026"""
1,"""458_083"""


genus_class,id
i64,str
0,"""445_057"""
0,"""445_060"""
0,"""446_057"""
0,"""446_060"""
0,"""459_037"""
1,"""441_032"""
1,"""446_052"""
1,"""458_062"""
1,"""459_026"""


In [26]:
id2split = {}
for i in train_ids:
    id2split[i] = "train"
for i in probe_train_ids:
    id2split[i] = "probe_train"
for i in probe_val_ids:
    id2split[i] = "probe_val"
meta_df = meta_df.with_columns(
    pl.col("id").replace(id2split).alias("split")
)
meta_df

organism,genus,data_file,peak_file,genus_class,id1,id2,grad_len,id,split
str,str,str,str,i64,str,str,str,str,str
"""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026""","""BBM_437_P110_31_MIA_026.mzML""",2,"""437""","""026""","""""","""437_026""","""train"""
"""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027""","""BBM_441_P110_31_MIA_027.mzML""",0,"""441""","""027""","""""","""441_027""","""probe_train"""
"""Micrococcus cohnii""","""Micrococcus""","""BBM_441_P110_31_MIA_032""","""BBM_441_P110_31_MIA_032.mzML""",1,"""441""","""032""","""""","""441_032""","""probe_val"""
"""Micrococcus lylae""","""Micrococcus""","""BBM_446_P110_31_MIA_052""","""BBM_446_P110_31_MIA_052.mzML""",1,"""446""","""052""","""""","""446_052""","""probe_val"""
"""Listeria seeligeri""","""Listeria""","""BBM_446_P110_31_MIA_057""","""BBM_446_P110_31_MIA_057.mzML""",0,"""446""","""057""","""""","""446_057""","""probe_val"""
…,…,…,…,…,…,…,…,…,…
"""Buttiauxella noackiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_010_10""","""BBM_436_P110_31_MIA_010_10.mzM…",3,"""436""","""010""","""10""","""436_010""","""train"""
"""Buttiauxella warmboldiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_011_10""","""BBM_436_P110_31_MIA_011_10.mzM…",3,"""436""","""011""","""10""","""436_011""","""train"""
"""Buttiauxella ferragutiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_013_10""","""BBM_436_P110_31_MIA_013_10.mzM…",3,"""436""","""013""","""10""","""436_013""","""train"""


In [27]:
split_df = meta_df.filter(pl.col("split") == "train")
print("Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

Total: 30


genus_class,count
i64,u32
2,9
3,21


genus_class,proportion
i64,f64
2,0.3
3,0.7


In [28]:
split_df = meta_df.filter(pl.col("split") == "probe_train")
print("Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

Total: 17


genus_class,count
i64,u32
0,8
1,9


genus_class,proportion
i64,f64
0,0.470588
1,0.529412


In [29]:
split_df = meta_df.filter(pl.col("split") == "probe_val")
print("Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

Total: 13


genus_class,count
i64,u32
0,7
1,6


genus_class,proportion
i64,f64
0,0.538462
1,0.461538


In [30]:
# train SpectrumDataset
train_df = pl.concat(
    [
        dfs[mzml_file] for mzml_file 
        in meta_df.filter(pl.col("split") == "train")["peak_file"].to_list()
    ], 
    how="vertical"
)
train_df = train_df.join(meta_df, on="peak_file", how="left")
train_df

peak_file,scan_id,ms_level,precursor_mz,precursor_charge,mz_array,intensity_array,organism,genus,data_file,genus_class,id1,id2,grad_len,id,split
str,str,u8,f64,i16,list[f64],list[f64],str,str,str,i64,str,str,str,str,str
"""BBM_437_P110_31_MIA_026.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[365.254059, 368.902618, … 1293.457764]","[0.1274, 0.129464, … 0.134766]","""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026""",2,"""437""","""026""","""""","""437_026""","""train"""
"""BBM_437_P110_31_MIA_026.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[369.93396, 379.935516, … 1260.21228]","[0.140476, 0.215298, … 0.168891]","""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026""",2,"""437""","""026""","""""","""437_026""","""train"""
"""BBM_437_P110_31_MIA_026.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[363.236572, 364.840363, … 885.577759]","[0.085937, 0.080867, … 0.072026]","""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026""",2,"""437""","""026""","""""","""437_026""","""train"""
"""BBM_437_P110_31_MIA_026.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[360.937775, 363.237213, … 828.537598]","[0.070572, 0.079136, … 0.095153]","""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026""",2,"""437""","""026""","""""","""437_026""","""train"""
"""BBM_437_P110_31_MIA_026.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[360.937469, 363.192413, … 924.883179]","[0.067324, 0.065806, … 0.074065]","""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026""",2,"""437""","""026""","""""","""437_026""","""train"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""BBM_436_P110_31_MIA_022_10.mzM…","""controllerType=0 controllerNum…",1,NaN,0,"[366.372528, 367.123779, … 1122.458496]","[0.075359, 0.177399, … 0.083036]","""Shigella flexneri""","""Shigella""","""BBM_436_P110_31_MIA_022_10""",2,"""436""","""022""","""10""","""436_022""","""train"""
"""BBM_436_P110_31_MIA_022_10.mzM…","""controllerType=0 controllerNum…",1,NaN,0,"[367.123657, 367.145294, … 1122.659912]","[0.190445, 0.516054, … 0.080165]","""Shigella flexneri""","""Shigella""","""BBM_436_P110_31_MIA_022_10""",2,"""436""","""022""","""10""","""436_022""","""train"""
"""BBM_436_P110_31_MIA_022_10.mzM…","""controllerType=0 controllerNum…",1,NaN,0,"[363.31012, 366.372406, … 1122.659424]","[0.082119, 0.092303, … 0.094978]","""Shigella flexneri""","""Shigella""","""BBM_436_P110_31_MIA_022_10""",2,"""436""","""022""","""10""","""436_022""","""train"""


In [31]:
# val SpectrumDataset
val_df = pl.concat(
    [
        dfs[mzml_file] for mzml_file 
        in meta_df.filter(
            pl.col("split").is_in(["probe_train", "probe_val"])
        )["peak_file"].to_list()
    ], 
    how="vertical"
)
val_df = val_df.join(meta_df, on="peak_file", how="left")
val_df

peak_file,scan_id,ms_level,precursor_mz,precursor_charge,mz_array,intensity_array,organism,genus,data_file,genus_class,id1,id2,grad_len,id,split
str,str,u8,f64,i16,list[f64],list[f64],str,str,str,i64,str,str,str,str,str
"""BBM_441_P110_31_MIA_027.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[362.9729, 364.892395, … 1227.718384]","[0.292306, 0.265438, … 0.262193]","""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027""",0,"""441""","""027""","""""","""441_027""","""probe_train"""
"""BBM_441_P110_31_MIA_027.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[360.183563, 360.686737, … 1299.078979]","[0.174292, 0.197958, … 0.185285]","""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027""",0,"""441""","""027""","""""","""441_027""","""probe_train"""
"""BBM_441_P110_31_MIA_027.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[360.183472, 360.685486, … 1236.184937]","[0.177232, 0.24495, … 0.150059]","""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027""",0,"""441""","""027""","""""","""441_027""","""probe_train"""
"""BBM_441_P110_31_MIA_027.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[360.685547, 362.8909, … 1245.137451]","[0.228129, 0.230556, … 0.144729]","""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027""",0,"""441""","""027""","""""","""441_027""","""probe_train"""
"""BBM_441_P110_31_MIA_027.mzML""","""controllerType=0 controllerNum…",1,NaN,0,"[360.685333, 360.937653, … 1048.561523]","[0.178574, 0.142133, … 0.194109]","""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027""",0,"""441""","""027""","""""","""441_027""","""probe_train"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""BBM_458_P110_31_MIA_083_10.mzM…","""controllerType=0 controllerNum…",1,NaN,0,"[363.177795, 363.215118, … 1171.582886]","[0.120142, 0.095812, … 0.126205]","""Micrococcus yunnanensis""","""Micrococcus""","""BBM_458_P110_31_MIA_083_10""",1,"""458""","""083""","""10""","""458_083""","""probe_train"""
"""BBM_458_P110_31_MIA_083_10.mzM…","""controllerType=0 controllerNum…",1,NaN,0,"[363.178009, 363.215851, … 1171.583984]","[0.130162, 0.115957, … 0.115233]","""Micrococcus yunnanensis""","""Micrococcus""","""BBM_458_P110_31_MIA_083_10""",1,"""458""","""083""","""10""","""458_083""","""probe_train"""
"""BBM_458_P110_31_MIA_083_10.mzM…","""controllerType=0 controllerNum…",1,NaN,0,"[363.178375, 363.215149, … 1171.582642]","[0.151227, 0.130142, … 0.154679]","""Micrococcus yunnanensis""","""Micrococcus""","""BBM_458_P110_31_MIA_083_10""",1,"""458""","""083""","""10""","""458_083""","""probe_train"""


In [32]:
train_stream = SpectrumDataset(
    train_df.select(["mz_array", "intensity_array", "genus_class"]), 
    batch_size=256, # streaming batch size, not training batch size
)
print("N train spectra", train_stream.n_spectra)
train_lance_path = str(train_stream.path)   # this is the key
print("Lance dataset path:", train_lance_path)
train_dataset = LanceMapDataset(train_lance_path, seq_len=config.data.max_num_peaks)

val_stream = SpectrumDataset(
    val_df.select(["mz_array", "intensity_array", "genus_class"]), 
    batch_size=256, # streaming batch size, not training batch size
)
print("N train spectra", val_stream.n_spectra)
val_lance_path = str(val_stream.path)   # this is the key
print("Lance dataset path:", val_lance_path)
val_dataset = LanceMapDataset(val_lance_path, seq_len=config.data.max_num_peaks)

N train spectra 87385
Lance dataset path: /tmp/tmpbllt6275/8f0957a3-32f8-4d8a-aab0-13e8351e05f9.lance
N train spectra 87112
Lance dataset path: /tmp/tmp366087e6/d520a1b4-31c0-43fe-927c-907a4a71403e.lance


In [33]:
run_labels = dict(zip(meta_df["peak_file"], meta_df["genus_class"]))

probe_train_dataset = RunDataset(
    [
        dfs[mzml_file] for mzml_file 
        in meta_df.filter(pl.col("split") == "probe_train")["peak_file"].to_list()
    ], 
    run_labels=run_labels, 
    seq_len=config.data.max_num_peaks
)
probe_val_dataset = RunDataset(
    [
        dfs[mzml_file] for mzml_file 
        in meta_df.filter(pl.col("split") == "probe_val")["peak_file"].to_list()
    ], 
    run_labels=run_labels,
    seq_len=config.data.max_num_peaks
)
len(probe_train_dataset), len(probe_val_dataset)

100%|██████████| 13/13 [00:00<00:00, 87.65it/s]


(17, 13)

In [34]:
def run_collate_fn(rows):
    # rows is a list of dicts, one per spectrum
    keys = rows[0].keys()
    batch = {}
    for key in keys:
        if key in ['mz_array', 'intensity_array']:
            batch[key] = [torch.tensor(r[key]) for r in rows]
        else:
            batch[key] = torch.tensor([r[key] for r in rows])
    return batch

In [35]:
BATCH_SIZE = config.data.batch_size

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=False
)
len(train_loader), len(val_loader)

(44, 44)

In [36]:
probe_train_loader = DataLoader(
    probe_train_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=True,
    collate_fn=run_collate_fn,
)
probe_val_loader = DataLoader(
    probe_val_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=False,
    collate_fn=run_collate_fn,
)
len(probe_train_loader), len(probe_val_loader)

(1, 1)

In [37]:
# class ConvRunAggregator(nn.Module):
#     def __init__(
#         self,
#         input_dim,
#         agg_type: str = "mean", # "mean", "conv-based"
#         n_temporal_steps: int = 100,
#     ):
#         self.input_dim = input_dim
#         self.agg_type = agg_type
#         self.n_temporal_steps = n_temporal_steps
        
#         self.temporal_pool = nn.AdaptiveAvgPool1d(self.n_temporal_steps) # (1, d, T) -> (1, d, n_t)
#         # aggregate & stack across bs: [(1, d, n_t)] x bs -> (bs, d, n_t)
#         # apply 1 temporal conv and then average across time (n_t)
#         self.aggregator = nn.Sequential(
#             nn.Conv1d(self.input_dim, self.input_dim, kernel_size=3, padding=1), # (bs, d, n_t) -> (bs, d, n_t)
#             nn.ReLU(inplace=True),
#             nn.AdaptiveAvgPool1d(1) # (1, d, n_t) -> (1, d, 1)
#         )

#     def forward(self, )

In [38]:
import torch
import torch.nn as nn
from torch import Tensor
from torchmetrics.functional import accuracy
# could instead use:
# self.train_accuracy = torchmetrics.classification.Accuracy(task="multiclass", num_classes=..., ignore_index=-1)
# self.val_accuracy = torchmetrics.classification.Accuracy(task="multiclass", num_classes=..., ignore_index=-1)


class OnlineFineTuner(L.Callback):
    def __init__(
        self,
        encoder_output_dim: int,
        num_classes: int,
        target_key: str,
        probe_train_loader,
        probe_val_loader=None,
        agg_type: str = "mean", # "mean", "conv-based"
        n_temporal_steps: int = 100,
        class_weights: Tensor | None = None,
        probe_lr: float = 1e-2,
    ) -> None:
        super().__init__()
        
        self.encoder_output_dim = encoder_output_dim
        self.num_classes = num_classes
        self.target_key = target_key
        self.agg_type = agg_type
        self.n_temporal_steps = n_temporal_steps
        self.class_weights = class_weights
        self.probe_lr = probe_lr

        if class_weights is not None:
            assert class_weights.size(0) == self.num_classes, "number of class weights must equal `num_classes`"

        self.probe_train_loader = probe_train_loader
        self.probe_val_loader = probe_val_loader

        self.optimizer: torch.optim.Optimizer
        # self.online_finetuner = nn.Linear(self.encoder_output_dim, self.num_classes).to(self.device)
        # self.optimizer = torch.optim.Adam(self.online_finetuner.parameters(), lr=self.probe_lr)

    def on_fit_start(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:
        # TODO: if self.agg_type == "conv_based": add conv aggregation layers?
        
        # finetuner layer becomes a part of the main (embedding) model
        pl_module.online_finetuner = nn.Linear(self.encoder_output_dim, self.num_classes).to(pl_module.device)
        # init optimizer inside the callback
        self.optimizer = torch.optim.Adam(pl_module.online_finetuner.parameters(), lr=self.probe_lr)

    def _encode_run(self, pl_module, run_mz, run_I):
        run_mz, run_I = run_mz.to(pl_module.device), run_I.to(pl_module.device)
        with torch.no_grad():
            peak_embs = pl_module.forward(run_mz, run_I)
            spec_embs = peak_embs.mean(dim=1) # average peak embeddings
            spec_embs = spec_embs.unsqueeze(dim=0) # (1, T, d)

            if self.agg_type == "mean":
                run_emb = spec_embs.mean(dim=1) # (1, d)
            # elif self.agg_type == "conv-based":
            #     spec_embs = spec_embs.permute(0, 2, 1) # (1, d, T)
            #     run_emb = self.temporal_pool(spec_embs) # (1, d, n_t)
        return run_emb

    def _step(self, pl_module, batch, train=True):
        # extract components from batch
        runs_mz = batch["mz_array"]
        runs_I = batch["intensity_array"]
        targets = batch[self.target_key].to(pl_module.device)

        runs_emb = [
            self._encode_run(pl_module, runs_mz[i], runs_I[i])
            for i in range(len(runs_mz))
        ]
        embs = torch.cat(runs_emb, dim=0) 
        print("DEBUG:", embs.size())

        # apply fine-tuner model
        # if self.agg_type == "conv-based":
        #     embs = self.aggregator(embs) # (bs, d, n_t) -> (bs, d, 1)
        #     embs = embs.squeeze(dim=-1) # (bs, d)
        preds = pl_module.online_finetuner(embs)

        # compute loss
        loss = F.cross_entropy(preds, targets, weight=self.class_weights)

        if train:
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()
        
        acc = accuracy(preds, targets, task="multiclass", num_classes=self.num_classes)

        # Display prediction sample
        n = 30
        sample_df = np.hstack((
            targets[:n].cpu().numpy()[:, None],
            F.softmax(preds, dim=1)[:n].detach().cpu().numpy(),
        ))
        sample_df = pd.DataFrame(
            sample_df, columns=[
                "targets", 
                "prob_0", 
                "prob_1", 
                # "prob_2", 
                # "prob_3", 
            ]
        )
        display(sample_df)
        return loss.detach(), acc.detach()

    def on_train_epoch_end(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:
        """
        Train the probe head for one pass over probe_train_loader, after SSL epoch ends.
        """
        was_training = pl_module.training
        pl_module.eval()
        pl_module.online_finetuner.train()

        total_loss = 0.0
        total_acc = 0.0
        n_batches = 0

        for batch in self.probe_train_loader:
            loss, acc = self._step(pl_module, batch, train=True)
            total_loss += float(loss)
            total_acc += float(acc)
            n_batches += 1

        avg_loss = total_loss / n_batches
        avg_acc = total_acc / n_batches

        # Log epoch metrics
        pl_module.log("online_train_loss", avg_loss, on_step=False, on_epoch=True, prog_bar=False)
        pl_module.log("online_train_acc", avg_acc, on_step=False, on_epoch=True, prog_bar=False)

        if was_training:
            pl_module.train()

    def on_validation_epoch_end(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:
        """
        Train the probe head for one pass over probe_train_loader, after SSL epoch ends.
        """
        was_training = pl_module.training
        pl_module.eval()
        pl_module.online_finetuner.eval()

        total_loss = 0.0
        total_acc = 0.0
        n_batches = 0

        with torch.no_grad():
            for batch in self.probe_val_loader:
                loss, acc = self._step(pl_module, batch, train=False)
                total_loss += float(loss)
                total_acc += float(acc)
                n_batches += 1

        avg_loss = total_loss / n_batches
        avg_acc = total_acc / n_batches

        # Log epoch metrics
        pl_module.log("online_val_loss", avg_loss, on_step=False, on_epoch=True, prog_bar=False)
        pl_module.log("online_val_acc", avg_acc, on_step=False, on_epoch=True, prog_bar=False)

        if was_training:
            pl_module.train()

In [39]:
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=2000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [40]:
model = MS1Encoder(
    d_model=config.model.d_model,
    nhead=config.model.nhead,
    dim_feedforward=config.model.dim_feedforward,
    n_layers=config.model.n_layers,
    dropout=config.model.dropout,
    n_bins=config.model.n_bins,
    bin_mz_min=config.model.bin_mz_min,
    bin_mz_max=config.model.bin_mz_max,
    masked_peaks_fraction=config.model.masked_peaks_fraction,
    lr=config.optimizer.lr,
    warmup_iters=config.optimizer.warmup_iters,
    cosine_schedule_period_iters=config.optimizer.cosine_schedule_period_iters,
)

In [41]:
root_dir = os.path.join(config.training.checkpoint_path, "foundation_model")
os.makedirs(root_dir, exist_ok=True)

logger = L.loggers.TensorBoardLogger(
    os.path.join(root_dir, "lightning_logs"),
    name=config.name,
)

In [42]:
# TODO: number of classes needs to be parametrized (based on number of classes in val subset)
# and validation classes need to be "renamed" (mapped) to always be 0 < C < n_classes
meta_df.filter(meta_df["id"].is_in(val_ids))["genus_class"].value_counts()

genus_class,count
i64,u32
1,15
0,15


In [43]:
online_finetuner = OnlineFineTuner(
    encoder_output_dim=config.model.d_model, 
    num_classes=2,#len(genus_class),
    target_key="label",
    probe_train_loader=probe_train_loader,
    probe_val_loader=probe_val_loader,
    agg_type='mean',
    class_weights=None,
    probe_lr=1e-3,#1e-2,
)
# checkpoint_callback = L.ModelCheckpoint(every_n_epochs=100, save_top_k=-1, save_last=True)

trainer = L.Trainer(
    logger=logger,
    default_root_dir=root_dir,
    callbacks=[online_finetuner], #checkpoint_callback],
    accelerator="gpu", #config.training.accelerator,
    devices=config.training.devices,
    max_epochs=500, #config.training.max_epochs,
    gradient_clip_val=config.training.gradient_clip_val,
    num_sanity_val_steps=2,
    log_every_n_steps=10,
)

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/mpominova/.local/share/mamba/envs/lcms-foundat ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
# Train the model
trainer.fit(
    model, 
    train_loader, 
    val_dataloaders=[val_loader],
    # ckpt_path=ckpt_path, # 
)


Validation DataLoader 0: 100%|██████████| 44/44 [00:34<00:00,  1.27it/s]DEBUG: torch.Size([13, 256])


,targets,prob_0,prob_1
0,1.0,0.472548,0.527452
1,1.0,0.458897,0.541103
2,0.0,0.493665,0.506335
3,0.0,0.491811,0.508189
4,1.0,0.483574,0.516425
5,0.0,0.478075,0.521925
6,1.0,0.468667,0.531333
7,0.0,0.467937,0.532063
8,0.0,0.475618,0.524382
9,1.0,0.480215,0.519785



Epoch 21: 100%|██████████| 44/44 [01:50<00:00,  0.40it/s, v_num=0, train_acc_mz_bin=0.0467, val_acc_mz_bin=0.0195]DEBUG: torch.Size([17, 256])


,targets,prob_0,prob_1
0,0.0,0.467812,0.532188
1,1.0,0.455653,0.544347
2,0.0,0.501057,0.498943
3,0.0,0.491144,0.508856
4,1.0,0.461757,0.538243
5,0.0,0.471082,0.528918
6,1.0,0.468072,0.531928
7,0.0,0.505401,0.494599
8,1.0,0.447159,0.552841
9,1.0,0.479194,0.520806


Epoch 22:  59%|█████▉    | 26/44 [00:41<00:28,  0.63it/s, v_num=0, train_acc_mz_bin=0.0481, val_acc_mz_bin=0.0195]